In [1]:
import glob
import json
import yaml
import shutil
import numpy as np
import pandas as pd
import datetime
from ray import tune
import hashlib
import rampwf as rw
import ramphy as rh
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
#import altair as alt
#alt.renderers.enable('default')
pd.options.display.max_columns = 200
pd.options.display.max_rows = 1000
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import DatetimeTickFormatter, Label
from bokeh.resources import CDN
from bokeh.embed import file_html
output_notebook()

kits_root = Path("/home/gpaolo/data/ramp-kits")

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.19) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


/tmp/ipykernel_2135175/2606921247.py:18: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


Loading BokehJS ...

In [2]:
def kaggle_select(kaggle_api, suffix, id):
    f_name = kaggle_api.string(getattr(id, "fileName"))
    select = f_name[5:5 + len(suffix)] == suffix
    select = select and kaggle_api.string(getattr(id, "publicScore")) != ''
    return select

def ramp_kit_plot(ramp_kit_dir):
    problem = rw.utils.assert_read_problem(ramp_kit_dir=ramp_kit_dir)
    score_names = [st.name for st in problem.score_types]
    score_type = problem.score_types[-1]
    valid_score_name = f'valid_{score_names[-1]}'
    metadata = json.load(open(Path(ramp_kit_dir) / "data" / "metadata.json"))
    kaggle_api = KaggleApi()
    kaggle_api.authenticate()
    
    action_f_names = glob.glob(f'{ramp_kit_dir}/actions/*')
    action_f_names.sort()
    ramp_program = []
    start_time = datetime.datetime(2024, 5, 8, 12, 30, 00)
    for action_f_name in action_f_names:
        f_name = Path(action_f_name).name
        if f_name > str(start_time):
            ramp_program.append(rh.actions.load_ramp_action(action_f_name))
    
    hyperopt_actions = [ra for ra in ramp_program if ra.start_time > start_time and ra.name == "hyperopt"]
    blend_actions = [ra for ra in ramp_program if ra.start_time > start_time and (ra.name == "blend" or ra.name == "bag_then_blend")]
    train_actions = [ra for ra in ramp_program if ra.start_time > start_time and ra.name == "train"]
    print(blend_actions[-1].__dict__)
    start_time = min([ra.start_time for ra in ramp_program if ra.name == "hyperopt"])
    stop_time = max([ra.stop_time for ra in ramp_program])
    print(f"{len(hyperopt_actions)} rounds done")
    
    color_dict = {
        "catboost": "#2ca02c",
        "lgbm": "#ff7f0e",
        "xgboost": "#9467bd",
    }
    tooltips = [
        ("submission", "@submission"),
        ("score", "@valid_score"),
        ("run time [s]", "@runtime"),
    ]
    p = figure(
        title=f"AutoDS on {ramp_kit_dir}", x_axis_label='action time',
        x_axis_type="datetime", y_axis_label=f"valid {score_names[-1]}",
        tooltips=tooltips, width=1000, height=500,
    )
    ### Kaggle private leaderboard percentile rank lines
    quantile_start = 0.5
    quantile_resolution = 0.05
    try:
        public_leaderboard_scores = np.load(Path(ramp_kit_dir) / "data" / "public_leaderboard_scores.npy")
    except:
        pass
    try:
        private_leaderboard_scores = np.load(Path(ramp_kit_dir) / "data" / "private_leaderboard_scores.npy")
        if score_type.is_lower_the_better:
            private_leaderboard_ranks = np.clip(np.arange(0, quantile_start + 0.0001, quantile_resolution), 0, 1)
        else:
            private_leaderboard_ranks = np.clip(np.arange(quantile_start, 1.01, quantile_resolution), 0, 1)
        private_leaderboard_quantiles = np.quantile(private_leaderboard_scores, private_leaderboard_ranks)
        for qi, quantile in enumerate(private_leaderboard_quantiles):
            if score_type.is_lower_the_better:
                percentile_rank = int((1 - private_leaderboard_ranks[qi]) * 100)
            else:
                percentile_rank = int(private_leaderboard_ranks[qi] * 100)
            source = pd.DataFrame({
                "submission": [percentile_rank, percentile_rank],
                "time": [start_time, stop_time],
                "valid_score": [quantile, quantile],
            })
            p.line("time", "valid_score", line_width=1, color="#d62728", alpha=0.5, source=source)
            rank_text = Label(
                x=stop_time, y=quantile, text=str(percentile_rank), text_color="#d62728", text_alpha=0.5,
                text_align="left", text_font_size="6px", text_baseline="middle")
            p.add_layout(rank_text)     
    except:
        try:
            if score_type.is_lower_the_better:
                public_leaderboard_ranks = np.clip(np.arange(0, quantile_start + 0.0001, quantile_resolution), 0, 1)
            else:
                public_leaderboard_ranks = np.clip(np.arange(quantile_start, 1.01, quantile_resolution), 0, 1)
            public_leaderboard_quantiles = np.quantile(public_leaderboard_scores, public_leaderboard_ranks)
            for qi, quantile in enumerate(public_leaderboard_quantiles):
                if score_type.is_lower_the_better:
                    percentile_rank = int((1 - public_leaderboard_ranks[qi]) * 100)
                else:
                    percentile_rank = int(public_leaderboard_ranks[qi] * 100)
                source = pd.DataFrame({
                    "submission": [percentile_rank, percentile_rank],
                    "time": [start_time, stop_time],
                    "valid_score": [quantile, quantile],
                })
                p.line("time", "valid_score", line_width=1, color="#1f77b4", alpha=0.5, source=source)
                rank_text = Label(
                    x=stop_time, y=quantile, text=str(percentile_rank), text_color="#1f77b4", text_alpha=0.5,
                    text_align="left", text_font_size="6px", text_baseline="middle")
                p.add_layout(rank_text)     
        except:
            pass
    
    ### Hyperopt race action segment
    for ra in hyperopt_actions:
        if len(ra.mean_scores) > 0:
            submission = ra.kwargs["submission"]
            source = pd.DataFrame({
                "action time": [ra.start_time, ra.stop_time],
                "valid_score": [ra.mean_score, ra.mean_score],
                "submission": [submission, submission],
                "runtime": [int(ra.runtime.total_seconds()), int(ra.runtime.total_seconds())],
            })
            p.line("action time", "valid_score", legend_label=submission, line_width=3, color=color_dict[submission], source=source)
    
    ### Blended score after each action
    source = pd.DataFrame({
        "action time": [ra.start_time for ra in blend_actions],
        "valid_score": [ra.blended_score for ra in blend_actions],
        "submission": [ra.kwargs["submissions"] for ra in blend_actions],
        "runtime": [ra.runtime.total_seconds() for ra in blend_actions],
    })
    p.line("action time", "valid_score", legend_label='blend', line_width=3, color="black", source=source)
    
    ### Kaggle public and private scores
    try:            
        kaggle_submission_ids = kaggle_api.competition_submissions(competition=metadata["kaggle_name"])
        kaggle_submission_ids = [id for id in kaggle_submission_ids
                                 if not "_best_" in kaggle_api.string(getattr(id, "fileName"))]
        kaggle_public_scores = [kaggle_api.string(getattr(id, "publicScore")) for id in kaggle_submission_ids
                                if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_private_scores = [kaggle_api.string(getattr(id, "privateScore")) for id in kaggle_submission_ids
                                 if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_action_times = [kaggle_api.string(getattr(id, "description")) for id in kaggle_submission_ids
                               if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_file_names = [kaggle_api.string(getattr(id, "fileName")) for id in kaggle_submission_ids
                             if kaggle_select(kaggle_api, kit_suffix, id)]
        source = pd.DataFrame({
            "action time": kaggle_action_times,
            "valid_score": kaggle_public_scores,
            "submission": kaggle_file_names,
        })
        source["action time"] = pd.to_datetime(source["action time"], format='mixed')
        p.line("action time", "valid_score", legend_label='public kaggle', line_width=3, color="#1f77b4", source=source)
        
        if "".join(kaggle_private_scores) != "":
            source = pd.DataFrame({
                "action time": kaggle_action_times,
                "valid_score": kaggle_private_scores,
                "submission": kaggle_file_names,
            })
            source["action time"] = pd.to_datetime(source["action time"], format='mixed')
            p.line("action time", "valid_score", legend_label='private kaggle', line_width=3, color="#d62728", source=source)
            
            final_time = max(source["action time"])
            final_score = float(source[source["action time"] == final_time]["valid_score"].to_numpy()[0])
            if score_type.is_lower_the_better:
                final_rank = np.less(float(final_score), private_leaderboard_scores).mean()
            else:
                final_rank = np.greater(float(final_score), private_leaderboard_scores).mean()
            final_rank_text = Label(
                x=final_time, y=final_score, text=str(round(100 * final_rank)), text_color="#d62728", text_alpha=1,
                text_align="left", text_font_size="18px", text_baseline="middle")
            p.add_layout(final_rank_text)
        else:
            final_time = max(source["action time"])
            final_score = float(source[source["action time"] == final_time]["valid_score"].to_numpy()[0])
            if score_type.is_lower_the_better:
                final_rank = np.less(float(final_score), public_leaderboard_scores).mean()
            else:
                final_rank = np.greater(float(final_score), public_leaderboard_scores).mean()
            final_rank_text = Label(
                x=final_time, y=final_score, text=str(round(100 * final_rank)), text_color="#1f77b4", text_alpha=1,
                text_align="left", text_font_size="18px", text_baseline="middle")
            p.add_layout(final_rank_text)     
    except Exception as e:
        print(e)
        pass
    
    ### Training actions after the end of the hyperopt race
    for ra in train_actions:
        submission = ra.kwargs["submission"]
        base_submission = submission[:-len("_hyperopt_0000000000")]
        if base_submission in color_dict.keys():
            source = pd.DataFrame({
                "action time": [ra.start_time, ra.stop_time],
                "valid_score": [ra.mean_score, ra.mean_score],
                "submission": [submission, submission],
                "runtime": [int(ra.runtime.total_seconds()), int(ra.runtime.total_seconds())],
            })
            p.line("action time", "valid_score", legend_label=base_submission, line_width=3, color=color_dict[base_submission], source=source)
    
    p.xaxis[0].formatter = DatetimeTickFormatter(hourmin = "%b%d %H:%M", minutes = "%H:%M", minsec = "%H:%M:%Ss")
    return p

In [3]:
ramp_kit = f"kaggle_synthanic"
version = "1_1"
numbers = ["X", "X_pca", "llama3", "X_llama3"]

for number in numbers:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)


{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': PosixPath('/home/gpaolo/data/ramp-kits/kaggle_synthanic_v1_1_nX'), 'submissions': ['xgboost_hyperopt_b76e29ca25', 'xgboost_hyperopt_e5dccbc893', 'xgboost_hyperopt_c7ff04c649', 'lgbm_hyperopt_aafc507477', 'xgboost_hyperopt_e5fd54bc93', 'xgboost_hyperopt_68ab0ebbc2'], 'fold_idxs': range(900, 931)}, 'start_time': datetime.datetime(2024, 7, 9, 12, 1, 4, 180700), 'stop_time': datetime.datetime(2024, 7, 9, 12, 1, 29, 924882), 'blended_score': 0.7791468503814093, 'contributivities': {'lgbm_hyperopt_aafc507477': 333, 'xgboost_hyperopt_e5fd54bc93': 333, 'xgboost_hyperopt_68ab0ebbc2': 333, 'xgboost_hyperopt_b76e29ca25': 0, 'xgboost_hyperopt_e5dccbc893': 0, 'xgboost_hyperopt_c7ff04c649': 0}}
100 rounds done


/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': PosixPath('/home/gpaolo/data/ramp-kits/kaggle_synthanic_v1_1_nX_pca'), 'submissions': ['lgbm_hyperopt_3ef274af1a', 'lgbm_hyperopt_f37cc9c2fb', 'lgbm_hyperopt_12205a8010', 'lgbm_hyperopt_698f05e697', 'lgbm_hyperopt_a8914b123b', 'lgbm_hyperopt_511037a71d', 'lgbm_hyperopt_59a736969c', 'lgbm_hyperopt_3460edeac6', 'lgbm_hyperopt_73a22e4c08'], 'fold_idxs': range(900, 931)}, 'start_time': datetime.datetime(2024, 7, 9, 12, 1, 56, 640842), 'stop_time': datetime.datetime(2024, 7, 9, 12, 2, 37, 665353), 'blended_score': 0.7820087208999802, 'contributivities': {'lgbm_hyperopt_a8914b123b': 333, 'lgbm_hyperopt_59a736969c': 333, 'lgbm_hyperopt_73a22e4c08': 333, 'lgbm_hyperopt_3ef274af1a': 0, 'lgbm_hyperopt_f37cc9c2fb': 0, 'lgbm_hyperopt_12205a8010': 0, 'lgbm_hyperopt_698f05e697': 0, 'lgbm_hyperopt_511037a71d': 0, 'lgbm_hyperopt_3460edeac6': 0}}
100 rounds done


/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': PosixPath('/home/gpaolo/data/ramp-kits/kaggle_synthanic_v1_1_nllama3'), 'submissions': ['lgbm_hyperopt_376acc2062', 'lgbm_hyperopt_7e18aac2a8', 'lgbm_hyperopt_34aa348f8c', 'lgbm_hyperopt_464fa8ba6b', 'lgbm_hyperopt_c43423e9fb', 'lgbm_hyperopt_89ae7b56a2', 'lgbm_hyperopt_82a786c28c', 'lgbm_hyperopt_698f417b24', 'lgbm_hyperopt_6a0166245c', 'lgbm_hyperopt_eaf08f26f6', 'lgbm_hyperopt_f785b50875'], 'fold_idxs': range(900, 931)}, 'start_time': datetime.datetime(2024, 7, 9, 12, 2, 38, 223968), 'stop_time': datetime.datetime(2024, 7, 9, 12, 3, 27, 874205), 'blended_score': 0.7794590544379807, 'contributivities': {'lgbm_hyperopt_f785b50875': 1000, 'lgbm_hyperopt_376acc2062': 0, 'lgbm_hyperopt_7e18aac2a8': 0, 'lgbm_hyperopt_34aa348f8c': 0, 'lgbm_hyperopt_464fa8ba6b': 0, 'lgbm_hyperopt_c43423e9fb': 0, 'lgbm_hyperopt_89ae7b56a2': 0, 'lgbm_hyperopt_82a786c28c': 0, 'lgbm_hyperopt_698f417b24': 0, 'lgbm_hyper

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': PosixPath('/home/gpaolo/data/ramp-kits/kaggle_synthanic_v1_1_nX_llama3'), 'submissions': ['xgboost_hyperopt_f1043e2a49', 'xgboost_hyperopt_afdc1ea47d', 'xgboost_hyperopt_438959de37', 'xgboost_hyperopt_3f3b30d2d8', 'xgboost_hyperopt_d0e493fe24', 'xgboost_hyperopt_7d4bc3b833'], 'fold_idxs': range(900, 931)}, 'start_time': datetime.datetime(2024, 7, 9, 12, 1, 30, 402642), 'stop_time': datetime.datetime(2024, 7, 9, 12, 1, 56, 121926), 'blended_score': 0.7787097647022093, 'contributivities': {'xgboost_hyperopt_438959de37': 1000, 'xgboost_hyperopt_f1043e2a49': 0, 'xgboost_hyperopt_afdc1ea47d': 0, 'xgboost_hyperopt_3f3b30d2d8': 0, 'xgboost_hyperopt_d0e493fe24': 0, 'xgboost_hyperopt_7d4bc3b833': 0}}
100 rounds done


2024-07-09 12:05:35,330 WARNING Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ProxyError('Cannot connect to proxy.', OSError('Tunnel connection failed: 407 Proxy Authentication Required'))': /api/v1/competitions/submissions/list/tabular-playground-series-apr-2021
2024-07-09 12:05:35,803 WARNING Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ProxyError('Cannot connect to proxy.', OSError('Tunnel connection failed: 407 Proxy Authentication Required'))': /api/v1/competitions/submissions/list/tabular-playground-series-apr-2021
2024-07-09 12:05:36,236 WARNING Retrying (Retry(total=0, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ProxyError('Cannot connect to proxy.', OSError('Tunnel connection failed: 407 Proxy Authentication Required'))': /api/v1/competitions/submissions/list/tabular-playground-series-apr-2021


HTTPSConnectionPool(host='www.kaggle.com', port=443): Max retries exceeded with url: /api/v1/competitions/submissions/list/tabular-playground-series-apr-2021 (Caused by ProxyError('Cannot connect to proxy.', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))


In [4]:
results_summary_df = pd.read_csv(kits_root / "results_summary.csv")
for col in results_summary_df.columns:
    if col[:14] == "contributivity" or col[:6] == "rounds":
        results_summary_df[col] = results_summary_df[col].fillna(0)
        results_summary_df[col] = results_summary_df[col].astype("int64")
    if col[:7] == "runtime":
        results_summary_df[col] = results_summary_df[col].astype("timedelta64[ns]")
for row_i, row in results_summary_df.iterrows():
    if row["kaggle_finished"] == 1:
        kit_suffix = f"v{row['version']}_n{row['number']}"
        ramp_kit_dir = f"{row['ramp_kit']}_{kit_suffix}"
        problem = rw.utils.assert_read_problem(ramp_kit_dir=kits_root/ ramp_kit_dir)
        score_type = problem.score_types[-1]
        final_cols = ["last_blend", "bagged_then_blended"]
        eval_col = "kaggle_public"
#        eval_col = "valid"
        eval_cols = [f"{eval_col}_{fc}" for fc in final_cols]
        if score_type.is_lower_the_better:
            best_public_submission = row[eval_cols].idxmin()[len(eval_col) + 1:]
        else:
            best_public_submission = row[eval_cols].idxmax()[len(eval_col) + 1:]
        if np.isnan(row[f"kaggle_private_prank_{best_public_submission}"]):
            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_public_prank_{best_public_submission}"]
#            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_public_prank_growing_folds"]
        else:
            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_private_prank_{best_public_submission}"]
#            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_private_prank_growing_folds"]
        results_summary_df.loc[row_i, "kaggle_private_prank_best_submission"] = best_public_submission
#results_summary_df["number"] = results_summary_df["number"].astype(int)
#results_summary_df["run_finished"] = 0
#results_summary_df["kaggle_finished"] = 0
results_summary_df

,ramp_kit,version,number,kaggle_private_prank_best_public,kaggle_private_prank_best_submission,server,run_finished,kaggle_finished,valid_last_blend,valid_growing_folds,valid_mean_lgbm,valid_bagged_lgbm,valid_mean_xgboost,valid_bagged_xgboost,valid_mean_catboost,valid_bagged_catboost,kaggle_private_last_blend,kaggle_private_growing_folds,kaggle_private_lgbm,kaggle_private_xgboost,kaggle_private_catboost,kaggle_private_prank_last_blend,kaggle_private_prank_growing_folds,kaggle_private_prank_lgbm,kaggle_private_prank_xgboost,kaggle_private_prank_catboost,kaggle_public_last_blend,kaggle_public_growing_folds,kaggle_public_lgbm,kaggle_public_xgboost,kaggle_public_catboost,kaggle_public_prank_last_blend,kaggle_public_prank_growing_folds,kaggle_public_prank_lgbm,kaggle_public_prank_xgboost,kaggle_public_prank_catboost,contributivity_last_blend_lgbm,contributivity_last_blend_xgboost,contributivity_last_blend_catboost,contributivity_growing_folds_lgbm,contributivity_growing_folds_xgboost,contributivity_growing_folds_catboost,runtime_hyperopt_xgboost,runtime_hyperopt_catboost,runtime_last_blend,runtime_growing_folds,rounds_hyperopt_lgbm,rounds_hyperopt_xgboost,rounds_hyperopt_catboost,kaggle_private_bagged_then_blended,kaggle_public_bagged_then_blended,kaggle_private_prank_bagged_then_blended,kaggle_public_prank_bagged_then_blended,valid_bagged_then_blended,contributivity_bagged_then_blended_lgbm,contributivity_bagged_then_blended_xgboost,contributivity_bagged_then_blended_catboost,runtime_hyperopt,runtime_hyperopt_lgbm
0,kaggle_synthanic,1_1,X,45.943775,last_blend,10.199.195.60,1,1,0.779397,0.779490,0.777984,0.778658,0.778123,0.778897,0.775268,0.776066,0.79035,0.78976,0.78914,0.78922,0.78124,45.943775,44.337349,41.686747,41.847390,19.036145,0.79345,0.79119,0.79030,0.78965,0.78513,56.706827,49.558233,45.943775,43.453815,28.755020,148,852,0,333,551,116,3 days 02:10:09.568934,0 days 08:25:40.338075,0 days 04:26:50.832185,0 days 04:08:27.932430,39,57,4,NaN,NaN,NaN,NaN,NaN,0,0,0,5 days 02:56:29.069073,1 days 16:20:39.162064
1,kaggle_synthanic,1_1,X_llama3,50.200803,last_blend,10.199.195.63,1,1,0.778949,0.779459,0.778319,0.778814,0.778377,0.778606,0.777110,0.777503,0.79142,0.79221,0.79134,0.79118,0.78218,50.200803,52.771084,49.879518,49.477912,21.445783,0.79377,0.79401,0.79413,0.79284,0.78303,57.188755,57.670683,58.152610,55.020080,23.614458,0,1000,0,333,519,148,2 days 12:51:43.216266,0 days 21:05:50.857401,0 days 10:03:03.872699,0 days 16:09:34.045402,3,87,10,NaN,NaN,NaN,NaN,NaN,0,0,0,3 days 16:28:13.129321,0 days 06:30:39.055654
2,kaggle_synthanic,1_1,X_pca,48.674699,last_blend,10.199.195.60,1,1,0.782748,0.781624,0.780252,0.780978,0.779761,0.780552,0.776981,0.777898,0.79093,0.79096,0.79032,0.79052,0.77749,48.674699,48.835341,45.943775,46.345382,11.405622,0.79478,0.79321,0.79268,0.79300,0.77807,59.277108,55.983936,54.216867,55.582329,12.449799,1001,0,0,690,251,59,0 days 04:46:25.949316,0 days 06:43:11.049002,0 days 02:03:49.200525,0 days 01:56:43.137736,90,7,3,NaN,NaN,NaN,NaN,NaN,0,0,0,0 days 23:10:16.089887,0 days 11:40:39.091569
3,kaggle_synthanic,1_1,llama3,40.240964,last_blend,10.199.195.60,1,1,0.780219,0.779740,0.778777,0.779178,0.776423,0.777294,0.774439,0.775619,0.78876,0.78871,0.78757,0.78968,0.77994,40.240964,40.000000,37.028112,43.534137,14.698795,0.78917,0.79014,0.78957,0.79256,0.78259,41.767068,45.461847,43.293173,54.056225,22.570281,1000,0,0,755,159,86,0 days 17:44:25.135549,0 days 23:35:24.614914,0 days 03:35:03.030303,0 days 06:41:47.652852,95,3,2,NaN,NaN,NaN,NaN,NaN,0,0,0,3 days 18:57:41.017584,2 days 01:37:51.267121


In [7]:
public_summary_df = results_summary_df[(results_summary_df["run_finished"] == 1) & (results_summary_df["kaggle_finished"] == 1)].drop(
    columns=["server", "kaggle_private_prank_best_submission", "version"]
    ).groupby(["ramp_kit", "number"]).mean()
public_summary_df["AutoDS"] = round(public_summary_df["kaggle_private_prank_best_public"])
public_summary_df["AutoDS"] = round(public_summary_df["AutoDS"].fillna(public_summary_df["kaggle_private_prank_best_public"]))
public_summary_df = public_summary_df[["AutoDS"]].astype(int).reset_index()
public_summary_df["Kaggle challenge"] = ""
for row_i, row in public_summary_df.iterrows():
    kit_suffix = f"v1_1_n{row['number']}"
    ramp_kit_dir = f"{row['ramp_kit']}_{kit_suffix}"
    metadata = json.load(open(kits_root / ramp_kit_dir / "data" / "metadata.json"))
    public_summary_df.loc[row_i, "Kaggle challenge"] = f"{metadata['title']} {metadata['prediction_type']}"  
public_summary_df = public_summary_df.set_index("ramp_kit")
weco_df = pd.read_csv(kits_root / "weco.csv").set_index("ramp_kit")
public_summary_df = public_summary_df.join(weco_df)
public_summary_df = public_summary_df[["Kaggle challenge", "number", "AutoDS", "kaggle_prank"]]
public_summary_df = public_summary_df.rename(columns={
    "Kaggle challenge": "Kaggle challenge", "number": "AutoDs version", "AutoDS": "AutoDS % rank", "kaggle_prank": "WecoAI % rank"})

In [8]:
public_summary_df

,Kaggle challenge,AutoDs version,AutoDS % rank,WecoAI % rank
ramp_kit,,,,
kaggle_synthanic,Kaggle synthanic binary classification,X,46,22
kaggle_synthanic,Kaggle synthanic binary classification,X_llama3,50,22
kaggle_synthanic,Kaggle synthanic binary classification,X_pca,49,22
kaggle_synthanic,Kaggle synthanic binary classification,llama3,40,22


In [14]:
results_summary_df["contributivity_bagged_then_blended_lgbm"] = np.nan
results_summary_df["contributivity_bagged_then_blended_xgboost"] = np.nan
results_summary_df["contributivity_bagged_then_blended_catboost"] = np.nan
cols = list(results_summary_df.columns)
results_summary_df = results_summary_df[cols[0:42] + cols[-3:] + cols[42:-3]]

/tmp/ipykernel_2135175/2828251909.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_summary_df["contributivity_bagged_then_blended_lgbm"] = np.nan
/tmp/ipykernel_2135175/2828251909.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_summary_df["contributivity_bagged_then_blended_xgboost"] = np.nan
/tmp/ipykernel_2135175/2828251909.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

In [16]:
results_summary_df[["ramp_kit", "runtime_hyperopt"]].groupby(["ramp_kit"]).mean().sort_values(by="runtime_hyperopt")

,runtime_hyperopt
ramp_kit,
kaggle_synthanic,3 days 09:23:09.826466250


In [17]:
results_summary_df["kaggle_private_best"] = np.nan
results_summary_df["kaggle_private_prank_best"] = np.nan
results_summary_df["kaggle_private_percentage_improvement"] = np.nan
results_summary_df["kaggle_private_prank_improvement"] = np.nan
results_summary_df["kaggle_private_percentage_improvement_btb_over_lb"] = np.nan
results_summary_df["kaggle_private_prank_improvement_btb_over_lb"] = np.nan
for row_i, row in results_summary_df.iterrows():
    if row["run_finished"] == 1:
        kit_suffix = f"v{row['version']}_n{row['number']}"
        ramp_kit_dir = f"{row['ramp_kit']}_{kit_suffix}"
        metadata = json.load(open(kits_root/ramp_kit_dir / "data" / "metadata.json"))
        problem = rw.utils.assert_read_problem(ramp_kit_dir=kits_root/ramp_kit_dir)
        score_type = problem.score_types[-1]
        if score_type.is_lower_the_better:
            results_summary_df.loc[row_i, "kaggle_private_best"] = min(
                row["kaggle_private_lgbm"], row["kaggle_private_xgboost"], row["kaggle_private_catboost"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement"] =\
                200 * (results_summary_df.loc[row_i, "kaggle_private_best"] - row["kaggle_private_last_blend"]) /\
                (row["kaggle_private_best"] + row["kaggle_private_last_blend"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement_btb_over_lb"] =\
                200 * (row["kaggle_private_last_blend"] - row["kaggle_private_bagged_then_blended"]) /\
                (row["kaggle_private_bagged_then_blended"] + row["kaggle_private_last_blend"])
        else:
            results_summary_df.loc[row_i, "kaggle_private_best"] = max(
                row["kaggle_private_lgbm"], row["kaggle_private_xgboost"], row["kaggle_private_catboost"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement"] =\
                200 * (row["kaggle_private_last_blend"] - row["kaggle_private_best"]) /\
                (results_summary_df.loc[row_i, "kaggle_private_best"] + row["kaggle_private_last_blend"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement_btb_over_lb"] =\
                200 * (row["kaggle_private_bagged_then_blended"] - row["kaggle_private_last_blend"]) /\
                (row["kaggle_private_bagged_then_blended"] + row["kaggle_private_last_blend"])
        results_summary_df.loc[row_i, "kaggle_private_prank_best"] = max(
                row["kaggle_private_prank_lgbm"], row["kaggle_private_prank_xgboost"], row["kaggle_private_prank_catboost"])
        results_summary_df.loc[row_i, "kaggle_private_prank_improvement"] =\
            row["kaggle_private_prank_last_blend"] -  row["kaggle_private_prank_best"]
        results_summary_df.loc[row_i, "kaggle_private_prank_improvement_btb_over_lb"] =\
            row["kaggle_private_prank_bagged_then_blended"] -  row["kaggle_private_prank_last_blend"]


In [18]:
p = figure(
    title=f"Relative improvement of blending over best Kaggle private scores", x_axis_label='Percentage',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-3, 3, 13)
hist, edges = np.histogram(results_summary_df["kaggle_private_percentage_improvement"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_percentage_improvement'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_percentage_improvement'].median(), 1)}%")
show(p)

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [ ]:
p = figure(
    title=f"Relative improvement of blending over best Kaggle private rank", x_axis_label='Percentage point',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-12, 32, 45)
hist, edges = np.histogram(results_summary_df["kaggle_private_prank_improvement"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_prank_improvement'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_prank_improvement'].median(), 1)}%")
show(p)

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [ ]:
p = figure(
    title=f"Relative improvement of bagged then blended over last blend Kaggle private scores", x_axis_label='Percentage',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-3, 3, 13)
hist, edges = np.histogram(results_summary_df["kaggle_private_percentage_improvement_btb_over_lb"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_percentage_improvement_btb_over_lb'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_percentage_improvement_btb_over_lb'].median(), 1)}%")
show(p)

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [ ]:
p = figure(
    title=f"Relative improvement of bagged then blended over last blend Kaggle private rank", x_axis_label='Percentage point',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-12, 32, 45)
hist, edges = np.histogram(results_summary_df["kaggle_private_prank_improvement_btb_over_lb"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_prank_improvement_btb_over_lb'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_prank_improvement_btb_over_lb'].median(), 1)}%")
show(p)

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
